# Pipeline MLOps con MLflow: Predicción y Clasificación de Calidad de Vinos

**Asignatura:** Ciencia de Datos  
**Institución:** Universidad de Cuenca  
**Integrantes:** Joseph Salamea & Xavier Peña  
**Dataset:** Wine Quality (UCI Machine Learning Repository, ID 186, Cortez et al., 2009)  
**Herramienta Principal:** MLflow (Open Source, v2.x)  
**Herramienta Comparativa:** Weights & Biases (W&B)  

---

## Estructura del Pipeline
1. **Configuración del Entorno y Librerías**
2. **Carga y Estructuración de Datos (Vino Tinto y Blanco)**
3. **Control de Calidad: Valores Nulos, Tipos y Consistencia**
4. **Análisis Estadístico Descriptivo**
5. **Distribución de Calidad y Formulación del Problema de Clasificación**
6. **Análisis de Correlación Fisicoquímica**
7. **Perfiles Fisicoquímicos: Vinos Tintos vs. Vinos Blancos**
8. **Detección de Patrones y Outliers**
9. **Preprocesamiento y Partición Estratificada (Train / Test)**
10. **Síntesis de Hallazgos y Decisiones para el Modelado MLOps**


## 1. Configuración del Entorno y Librerías

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Configuración de estilos visuales y advertencias
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Fijación de semilla aleatoria para garantizar reproducibilidad determinista
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas y entorno configurado con reproducibilidad fijada.")


## 2. Carga y Estructuración de Datos

El conjunto de datos original de Cortez et al. (2009) contiene muestras correspondientes a variantes de vino verde portugués (*Vinho Verde*):
* **Vino Tinto (`winequality-red.csv`):** 1.599 instancias.
* **Vino Blanco (`winequality-white.csv`):** 4.898 instancias.

Ambas fuentes comparten 11 variables fisicoquímicas continuas y una variable sensorial objetivo (`quality`, puntaje de 0 a 10). Siguiendo las mejores prácticas, incorporamos un atributo indicativo `wine_type` y concatenamos ambas variantes en un dataset consolidado de **6.497 instancias**.


In [ ]:
# Definir rutas relativas con soporte para ejecución desde /notebooks o desde la raíz
path_red = os.path.join('..', 'data', 'winequality-red.csv')
path_white = os.path.join('..', 'data', 'winequality-white.csv')

if not os.path.exists(path_red):
    path_red = os.path.join('data', 'winequality-red.csv')
    path_white = os.path.join('data', 'winequality-white.csv')

# Cargar datasets con separador ';'
df_red = pd.read_csv(path_red, sep=';')
df_white = pd.read_csv(path_white, sep=';')

# Agregar variable categórica de tipo de vino
df_red['wine_type'] = 'red'
df_white['wine_type'] = 'white'

# Concatenar en DataFrame unificado
df = pd.concat([df_red, df_white], ignore_index=True)

print("Carga de datos completada:")
print(f" - Vinos Tintos:  {df_red.shape[0]} filas, {df_red.shape[1]} columnas")
print(f" - Vinos Blancos: {df_white.shape[0]} filas, {df_white.shape[1]} columnas")
print(f" - Dataset Total: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()


## 3. Control de Calidad: Valores Nulos, Tipos y Consistencia

Verificamos la integridad de los datos para garantizar que no existan valores nulos, registros corruptos o tipos incompatibles que alteren las fases posteriores del pipeline.


In [ ]:
# Inspección de tipos de datos, nulos y cardinalidad
data_audit = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})

print("Auditoría de Integridad del Dataset:")
print(data_audit)

# Evaluación de filas duplicadas en variables fisicoquímicas
num_duplicates = df.duplicated().sum()
print(f"\nTotal de registros con características fisicoquímicas idénticas: {num_duplicates} ({(num_duplicates / len(df))*100:.2f}%)")
print("Nota: En enología, es factible que diferentes catas o lotes compartan métricas de laboratorio idénticas, por lo que se preservan para respetar la distribución empírica.")


## 4. Análisis Estadístico Descriptivo

Analizamos los estadísticos descriptivos (media, desviación estándar, rango intercuartílico, mínimos y máximos) para entender la dispersión y escala de cada variable fisicoquímica.


In [ ]:
# Resumen estadístico numérico redondeado
numeric_cols = df.select_dtypes(include=[np.number]).columns
stats_summary = df[numeric_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats_summary.round(3)


## 5. Distribución de Calidad y Formulación del Problema de Clasificación

La variable sensorial `quality` varía de 3 a 9 en los datos observados (no hay registros con 0, 1, 2 ni 10).
Conforme a la literatura estándar sobre el dataset de Cortez et al. (2009):
* La gran mayoría de los vinos se concentran en las categorías medias (5 y 6).
* Formulamos un problema de **clasificación binaria**:
  * **Clase 1 (Buena Calidad):** `quality >= 6` (vinos calificados como buenos o excepcionales).
  * **Clase 0 (Calidad Regular / Baja):** `quality < 6` (vinos calificados con puntajes 3, 4 o 5).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución de puntuaciones discretas
sns.countplot(data=df, x='quality', hue='wine_type', palette=['#722F37', '#E6C687'], ax=axes[0])
axes[0].set_title('Distribución de Puntuaciones Sensoriales (3 a 9) por Tipo', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Puntuación de Calidad (quality)')
axes[0].set_ylabel('Número de Muestras')
axes[0].legend(title='Tipo de Vino', labels=['Tinto', 'Blanco'])

# 2. Creación y distribución de la variable binaria
df['good_quality'] = (df['quality'] >= 6).astype(int)

sns.countplot(data=df, x='good_quality', hue='wine_type', palette=['#722F37', '#E6C687'], ax=axes[1])
axes[1].set_title('Balance de Clases: Regular (<6) vs. Buena Calidad (>=6)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Target Binario (good_quality)')
axes[1].set_ylabel('Número de Muestras')
axes[1].set_xticklabels(['Regular / Baja (<6)', 'Buena Calidad (>=6)'])
axes[1].legend(title='Tipo de Vino', labels=['Tinto', 'Blanco'])

plt.tight_layout()
plt.show()

# Conteo y proporciones exactas
counts = df['good_quality'].value_counts()
props = df['good_quality'].value_counts(normalize=True) * 100
print("Balance del Target Binario (good_quality):")
print(f" - Clase 1 (Buena Calidad >= 6): {counts[1]} vinos ({props[1]:.2f}%)")
print(f" - Clase 0 (Regular / Baja < 6):  {counts[0]} vinos ({props[0]:.2f}%)")


## 6. Análisis de Correlación Fisicoquímica

Evaluamos la matriz de correlación lineal de Pearson entre todas las variables predictoras y la calidad. Esto permite identificar:
1. Qué atributos fisicoquímicos tienen mayor correlación directa o inversa con la calidad del vino.
2. Si existen variables fuertemente colineales entre sí (ej. densidad vs. alcohol, dióxido de azufre libre vs. total).


In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()

# Máscara para la parte superior del triángulo
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='vlag', vmin=-0.8, vmax=0.8,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación Fisicoquímica de Pearson (Vinho Verde)', fontsize=14, pad=12, fontweight='bold')
plt.show()

# Ranking de correlación contra la calidad
quality_corr = corr['quality'].drop(['quality', 'good_quality']).sort_values(ascending=False)
print("Correlación de atributos fisicoquímicos con 'quality':")
print(quality_corr.round(4))


## 7. Perfiles Fisicoquímicos: Vinos Tintos vs. Vinos Blancos

El vino tinto y el vino blanco difieren en sus procesos de fermentación y maceración con la piel de la uva. Comparamos las cuatro variables químicas con mayor divergencia:
* `alcohol`
* `volatile acidity` (acidez volátil)
* `residual sugar` (azúcar residual)
* `total sulfur dioxide` (dióxido de azufre total)


In [ ]:
features_compare = ['alcohol', 'volatile acidity', 'residual sugar', 'total sulfur dioxide']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(features_compare):
    sns.boxplot(data=df, x='wine_type', y=col, palette=['#722F37', '#E6C687'], ax=axes[i], width=0.4)
    axes[i].set_title(f'Distribución de {col.title()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Tipo de Vino')
    axes[i].set_xticklabels(['Tinto', 'Blanco'])
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

# Comparación numérica de medias
mean_compare = df.groupby('wine_type')[features_compare].mean().T
mean_compare.columns = ['Vino Tinto (Media)', 'Vino Blanco (Media)']
mean_compare['Diferencia Absoluta'] = (mean_compare['Vino Blanco (Media)'] - mean_compare['Vino Tinto (Media)']).abs()
print("Comparativa de Medias Fisicoquímicas por Tipo:")
print(mean_compare.round(3))


## 8. Detección de Patrones y Relación con la Calidad

Visualizamos cómo interactúan los dos principales factores correlacionados (`alcohol` y `volatile acidity`) con la calidad binaria del vino mediante gráficos de distribución y violines.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de violín: Alcohol vs Calidad
sns.violinplot(data=df, x='good_quality', y='alcohol', hue='wine_type', split=True,
               palette=['#722F37', '#E6C687'], ax=axes[0])
axes[0].set_title('Alcohol por Calidad y Tipo de Vino', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Calidad del Vino')
axes[0].set_xticklabels(['Regular (<6)', 'Buena (>=6)'])
axes[0].set_ylabel('Alcohol (% vol)')
axes[0].legend(title='Tipo', labels=['Tinto', 'Blanco'])

# Gráfico de violín: Acidez Volátil vs Calidad
sns.violinplot(data=df, x='good_quality', y='volatile acidity', hue='wine_type', split=True,
               palette=['#722F37', '#E6C687'], ax=axes[1])
axes[1].set_title('Acidez Volátil por Calidad y Tipo de Vino', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Calidad del Vino')
axes[1].set_xticklabels(['Regular (<6)', 'Buena (>=6)'])
axes[1].set_ylabel('Acidez Volátil (g/dm³)')
axes[1].legend(title='Tipo', labels=['Tinto', 'Blanco'])

plt.tight_layout()
plt.show()


## 9. Preprocesamiento y Partición Estratificada (Train / Test)

Para alimentar los experimentos de MLflow y evitar fuga de información (*data leakage*):
1. **Codificación:** Transformamos `wine_type` en una variable binaria numérica (`is_white_wine = 1` para blanco, `0` para tinto).
2. **Definición de Matrices:** Separamos la matriz de variables predictoras $X$ (12 variables) y el vector de etiquetas $y$ (`good_quality`).
3. **Partición Estratificada:** Dividimos el dataset en **80% Entrenamiento (Train)** y **20% Prueba (Test)** garantizando idéntica proporción de clases positivas mediante `stratify=y`.


In [ ]:
# 1. Codificación de variables
df_processed = df.copy()
df_processed['is_white_wine'] = (df_processed['wine_type'] == 'white').astype(int)

# 2. Selección de columnas de características predictoras
feature_cols = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'is_white_wine'
]

X = df_processed[feature_cols]
y = df_processed['good_quality']

# 3. Partición estratificada Train (80%) y Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Dimensiones de los conjuntos preparados para Modelado MLOps:")
print(f" - X_train: {X_train.shape} (80%) | y_train: {y_train.shape}")
print(f" - X_test:  {X_test.shape} (20%) | y_test:  {y_test.shape}")
print("\nVerificación de distribución de clases (Estratificación exitosa):")
print(f" - Train ratio vinos buenos (clase 1): {y_train.mean():.4f} ({y_train.sum()} muestras)")
print(f" - Test ratio vinos buenos (clase 1):  {y_test.mean():.4f} ({y_test.sum()} muestras)")


## 10. Síntesis de Hallazgos y Decisiones para el Modelado MLOps

A partir del análisis exploratorio de datos, fundamentamos las siguientes decisiones técnicas para la suite de experimentos:

| Hallazgo en el EDA | Impacto / Decisión para el Modelado |
| :--- | :--- |
| **Alcohol (+0.44 correlación):** Factor más determinante de alta calidad sensorial. | Será una variable con alta importancia de características (*feature importance*) en árboles y coeficientes positivos en modelos lineales. |
| **Acidez Volátil (-0.27 correlación):** Asociada a defectos del vino (olor a vinagre/ácido acético). | Crucial para penalizar vinos de baja calidad. |
| **Diferencias Marcadas Tinto vs. Blanco:** Vinos blancos tienen 3x más dióxido de azufre y azúcar residual; tintos tienen mayor acidez volátil. | Incluir `is_white_wine` como predictor permite al modelo ajustar los umbrales químicos según la variante del vino. |
| **Escalas Heterogéneas:** Variables en rangos de 0.01 a 280 (ej. cloruros vs dióxido de azufre total). | **Obligatorio escalar (`StandardScaler`)** para modelos sensibles a la escala como Regresión Logística y Ridge. Los modelos de ensamble basados en árboles (Random Forest y XGBoost) son invariantes a la escala monótona. |
| **Balance Moderado de Clases:** 63.3% clase positiva (>=6) vs. 36.7% clase regular (<6). | Emplearemos **F1-Score (Macro)**, **ROC-AUC** y **Accuracy** como métricas principales registradas en MLflow para evaluar el trade-off precisión/recall. |

---

## 11. Próxima Etapa: Seguimiento de Experimentos con MLflow

El conjunto de datos procesado `(X_train, X_test, y_train, y_test)` queda listo para ejecutar la configuración de MLflow:
1. Inicialización del servidor de tracking y creación del experimento `wine-quality-mlops`.
2. Ejecución y registro estructurado de **10+ experimentos**:
   * Experimentos 1-3: Regresión Logística y Ridge con diferentes penalizaciones (`C`, `alpha`) y escalado estándar.
   * Experimentos 4-7: Random Forest variando `n_estimators`, `max_depth` y `min_samples_split`.
   * Experimentos 8-10+: XGBoost Classifier con búsqueda de tasa de aprendizaje (`learning_rate`), `max_depth` y regularización.
3. Comparación de curvas ROC, matrices de confusión y métricas en la UI de MLflow (`mlflow ui`).
4. Selección y promoción formal del mejor modelo al **MLflow Model Registry** en estado `Production`.
5. Ejecución del benchmark comparativo con **Weights & Biases (W&B)** midiendo sobrecarga de latencia y autonomía offline.
